# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("J3_Analytics_SQL_Gold").getOrCreate()

SILVER_MAIN_PATH = "/lakehouse/silver/transactions"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b2e6b9c6-0632-4994-b7d9-40f12b534b88;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 155ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

Lignes disponibles dans v_silver : 12635228


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
df_dist_summary = spark.sql("""
    SELECT
        COUNT(*) AS total_transactions,
        ROUND(AVG(amount_usd), 2) AS moyenne_usd,
        ROUND(STDDEV(amount_usd), 2) AS ecart_type_usd,
        ROUND(SKEWNESS(amount_usd), 2) AS skewness,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.25), 2) AS p25_usd,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.50), 2) AS mediane_p50_usd,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.75), 2) AS p75_usd,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.75) - PERCENTILE_APPROX(amount_usd, 0.25), 2) AS iqr_usd,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.90), 2) AS p90_usd,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.95), 2) AS p95_usd,
        ROUND(MAX(amount_usd), 2) AS max_usd
    FROM v_silver
""")

assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)


+------------------+-----------+--------------+--------+-------+---------------+-------+-------+-------+-------+-------+
|total_transactions|moyenne_usd|ecart_type_usd|skewness|p25_usd|mediane_p50_usd|p75_usd|iqr_usd|p90_usd|p95_usd|max_usd|
+------------------+-----------+--------------+--------+-------+---------------+-------+-------+-------+-------+-------+
|12635228          |50.6       |74.04         |8.1     |11.08  |31.92          |66.4   |55.32  |108.46 |149.72 |6820.2 |
+------------------+-----------+--------------+--------+-------+---------------+-------+-------+-------+-------+-------+



**Notre justification statistique :**
> `L'analyse univariée des 12 635 228 transactions met en évidence une distribution fortement asymétrique vers la droite, confirmée par un coefficient d'asymétrie (SKEWNESS) très élevé de 8,1. En raison de cette étendue vers les valeurs extrêmes — où le 95e centile (P95) atteint 149,72 $ et le maximum    culmine  à 6 820,20 $ — la moyenne arithmétique (50,60 $) ainsi que l'écart-type (74,04 $) sont artificiellement gonflés et ne reflètent pas le montant d'une transaction courante. À l'inverse, la médiane (P50) s'établit à 31,92 $, révélant que la moitié des paiements sont inférieurs à ce seuil, tandis que 50 % des transactions centrales se concentrent dans un écart interquartile (IQR = P75 - P25) de 55,32 $ (entre 11,08 $ et 66,40 $). En conséquence, le couple Médiane + IQR est retenu comme indicateur de référence pour le tableau de bord décisionnel afin de garantir une lecture robuste, insensible aux valeurs aberrantes.`


---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:
GOLD_PARETO_PATH = "/lakehouse/gold/pareto_mcc"

df_gold_1 = spark.sql("""
    WITH agg_mcc AS (
        SELECT
            mcc,
            COUNT(*) AS nb_transactions,
            ROUND(SUM(amount_usd), 2) AS volume_total_usd,
            ROUND(PERCENTILE_APPROX(amount_usd, 0.50), 2) AS mediane_usd,
            ROUND(PERCENTILE_APPROX(amount_usd, 0.75) - PERCENTILE_APPROX(amount_usd, 0.25), 2) AS iqr_usd
        FROM v_silver
        GROUP BY mcc
    )
    SELECT
        DENSE_RANK() OVER (ORDER BY volume_total_usd DESC) AS rang_volume,
        mcc,
        nb_transactions,
        volume_total_usd,
        mediane_usd,
        iqr_usd,
        ROUND(
            (SUM(volume_total_usd) OVER (ORDER BY volume_total_usd DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / SUM(volume_total_usd) OVER ()) * 100, 2
        ) AS pct_cumule_pareto
    FROM agg_mcc
    ORDER BY rang_volume ASC
""")

assert df_gold_1 is not None, "TODO : Écrivez la requête SQL avec fenêtre de Pareto dans `df_gold_1`."

(
    df_gold_1.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PARETO_PATH)
)
df_gold_1.show(15, truncate=False)


26/11/27 10:15:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 1

+-----------+----+---------------+----------------+-----------+-------+-----------------+
|rang_volume|mcc |nb_transactions|volume_total_usd|mediane_usd|iqr_usd|pct_cumule_pareto|
+-----------+----+---------------+----------------+-----------+-------+-----------------+
|1          |4829|578530         |5.315851564E7   |80.0       |60.0   |8.31             |
|2          |5541|1135638        |5.123975542E7   |40.55      |62.59  |16.33            |
|3          |5411|1592560        |4.097076706E7   |11.46      |24.87  |22.74            |
|4          |5300|601928         |3.769756871E7   |51.79      |61.93  |28.63            |
|5          |5499|1170328        |3.735264605E7   |8.9        |61.57  |34.48            |
|6          |5912|772895         |3.511356687E7   |30.39      |56.76  |39.97            |
|7          |4900|242990         |2.765005313E7   |110.17     |87.51  |44.29            |
|8          |5311|475371         |2.703199066E7   |45.51      |51.51  |48.52            |
|9        

26/11/27 10:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


**Notre justification statistique :**
> `Analyse du classement métier et de la concentration de Pareto :
L'agrégation par code catégorie marchand (mcc) et le calcul du pourcentage cumulé mettent en évidence une forte concentration des flux financiers selon le principe de Pareto. À eux seuls, les 15 premiers codes marchands génèrent près de 70 % (69,91 %) du volume financier total. En tête du classement figurent les transferts de fonds (MCC 4829 avec 53,16 M$ et une médiane élevée de 80,00 $) ainsi que les stations-service (MCC 5541 avec 51,24 M$), suivis par la grande distribution alimentaire (MCC 5411 avec 40,97 M$), qui domine en revanche en nombre d'opérations (1,59 million de transactions) mais avec un panier médian nettement plus faible (11,46 $).`

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
GOLD_TIME_PATH = "/lakehouse/gold/evolution_mensuelle"

df_gold_2 = spark.sql("""
    WITH mensuel AS (
        SELECT
            tx_year_month AS mois,
            COUNT(*) AS nb_transactions,
            ROUND(SUM(amount_usd), 2) AS volume_mensuel_usd,
            ROUND(PERCENTILE_APPROX(amount_usd, 0.50), 2) AS mediane_mensuelle_usd,
            ROUND(AVG(has_error) * 100, 2) AS taux_erreur_pct
        FROM v_silver
        GROUP BY tx_year_month
    )
    SELECT
        mois,
        nb_transactions,
        volume_mensuel_usd,
        mediane_mensuelle_usd,
        taux_erreur_pct,
        LAG(volume_mensuel_usd, 1) OVER (ORDER BY mois) AS volume_mois_precedent,
        ROUND(
            ((volume_mensuel_usd - LAG(volume_mensuel_usd, 1) OVER (ORDER BY mois))
            / NULLIF(LAG(volume_mensuel_usd, 1) OVER (ORDER BY mois), 0)) * 100, 2
        ) AS croissance_mom_pct
    FROM mensuel
    ORDER BY mois ASC
""")

assert df_gold_2 is not None, "TODO : Écrivez la requête SQL temporelle avec LAG() dans `df_gold_2`."

(
    df_gold_2.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TIME_PATH)
)
df_gold_2.show(20, truncate=False)


26/11/27 10:16:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:16:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:17:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:17:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:17:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:17:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 1

+-------+---------------+------------------+---------------------+---------------+---------------------+------------------+
|mois   |nb_transactions|volume_mensuel_usd|mediane_mensuelle_usd|taux_erreur_pct|volume_mois_precedent|croissance_mom_pct|
+-------+---------------+------------------+---------------------+---------------+---------------------+------------------+
|2010-01|95865          |4905493.51        |32.27                |1.56           |NULL                 |NULL              |
|2010-02|88552          |4595271.24        |32.76                |1.54           |4905493.51           |-6.32             |
|2010-03|97947          |5069357.31        |32.77                |1.54           |4595271.24           |10.32             |
|2010-04|94829          |4944804.88        |32.93                |1.48           |5069357.31           |-2.46             |
|2010-05|99314          |5164323.25        |32.84                |1.57           |4944804.88           |4.44              |
|2010-06

26/11/27 10:17:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/27 10:17:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


**Analyse temporelle :**
> `Analyse de la dynamique temporelle et des variations saisonnières :
L'agrégation mensuelle couplée à la fonction de fenêtrage LAG() met en évidence une activité financière très régulière, oscillant autour de 5 millions de dollars par mois avec un panier médian remarquablement stable aux alentours de 32,00 $ à 32,99 $ et un taux d'erreur contenu autour de 1,55 %. L'indicateur de croissance mensuelle (croissance_mom_pct) révèle toutefois un effet calendaire récurrent au mois de février (baisse mécanique de -6,32 % en février 2010 et -10,48 % en février 2011 liée au nombre réduit de jours ouvrés), immédiatement compensée par un fort rebond en mars (+10,32 % en 2010 et +12,84 % en 2011), ainsi qu'une tendance globale haussière d'une année sur l'autre.`

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [5]:
GOLD_TOP_STATE_PATH = "/lakehouse/gold/top_mcc_par_etat"

df_gold_3 = spark.sql("""
    WITH state_mcc AS (
        SELECT
            merchant_state AS etat,
            mcc,
            COUNT(*) AS nb_transactions,
            ROUND(SUM(amount_usd), 2) AS volume_usd,
            ROUND(PERCENTILE_APPROX(amount_usd, 0.50), 2) AS mediane_usd
        FROM v_silver
        GROUP BY merchant_state, mcc
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (PARTITION BY etat ORDER BY volume_usd DESC) AS rn
        FROM state_mcc
    )
    SELECT
        etat,
        mcc AS top_mcc,
        nb_transactions,
        volume_usd,
        mediane_usd
    FROM ranked
    WHERE rn = 1
    ORDER BY volume_usd DESC
""")

(
    df_gold_3.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TOP_STATE_PATH)
)
df_gold_3.show(15, truncate=False)

# Compaction des fichiers Delta de la couche Silver
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)


+------+-------+---------------+-------------+-----------+
|etat  |top_mcc|nb_transactions|volume_usd   |mediane_usd|
+------+-------+---------------+-------------+-----------+
|ONLINE|4784   |674125         |2.385955327E7|34.26      |
|CA    |4829   |78724          |7299186.61   |80.0       |
|TX    |4829   |49976          |4765000.0    |100.0      |
|NY    |4829   |38673          |4068740.0    |100.0      |
|FL    |4829   |40807          |4046270.72   |100.0      |
|PA    |4829   |31785          |3149040.78   |100.0      |
|HI    |3730   |9689           |2438783.13   |231.19     |
|MI    |3771   |9689           |2420549.21   |229.0      |
|NV    |3775   |9518           |2402891.32   |232.58     |
|TN    |4829   |20362          |2198140.0    |120.0      |
|OH    |5541   |50560          |2135171.27   |38.62      |
|NC    |4829   |22736          |2118967.37   |80.0       |
|IL    |4829   |21831          |1898060.0    |80.0       |
|WA    |5541   |34666          |1689530.04   |51.18     

+--------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                              |metrics                                                                                                                                                                                                 |
+--------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/transactions|{1, 29, {289109926, 289109926, 2.89109926E8, 1, 289109926}, {1715209, 15976124, 1.0100804103448275E7, 29, 292923319}, 1, NULL, NULL, 1, 29, 0, false, 0, 0, 1795774818607, 0, 16, 0, NULL,

**Analyse géographique :**
> `Analyse géographique par État et compaction physique (OPTIMIZE) :
Le partitionnement par État via la fonction ROW_NUMBER() met en évidence des profils de consommation très contrastés : les transactions dématérialisées (ONLINE) dominent largement en volume (23,86 M$ sur le code MCC 4784 lié aux péages et frais de transport), tandis que les grands États comme la Californie (CA), le Texas (TX) ou New York (NY) sont portés par les transferts de fonds (MCC 4829), et que des destinations touristiques comme Hawaï (HI) ou le Nevada (NV) se distinguent par des paniers médians très élevés (~231 $ sur l'hôtellerie). Par ailleurs, l'exécution de la commande OPTIMIZE sur la table Silver a permis de fusionner 29 micro-fichiers Parquet en 1 seul fichier compact de ~289 Mo. Cette opération résout le Small Files Problem sur HDFS en réduisant le nombre d'objets suivis en mémoire vive (RAM) par le NameNode et en accélérant les lectures analytiques ultérieures.`